In [1]:
import ssl

ssl._create_default_https_context = ssl._create_unverified_context
import nltk
import os
import string
import numpy as np
import copy 
import pickle
import re
import math

import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from collections import Counter
from num2words import num2words


from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer

data = pd.read_csv("<your path for prepared isw dataset>")

In [2]:
data.head()

,Unnamed: 0,id,date,short_url,title,text_title,full_url,html_content,html_content_v6
0,1201,1202,2022-02-24,2022_02_24,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar..."
1,1366,1367,2022-02-25,2022_02_25,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar..."
2,181,182,2022-02-26,2022_02_26,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar..."
3,197,198,2022-02-27,2022_02_27,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar..."
4,1106,1107,2022-02-28,2022_02_28,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar..."


Functions preparing

In [3]:
def remove_one_letter_word(data):
    words = word_tokenize(str(data))
    
    new_text = ""
    for w in words:
        if (len(w) > 1):
            new_text = new_text + " " + w
            
    return new_text

In [4]:
def convert_lower_case(data):
    return np.char.lower(data)

In [5]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
print(stop_words)

{'have', 'be', "you've", 'again', 'for', 'at', 've', 'before', "i'd", 'having', 'o', 'of', 'been', 'me', 'ma', "couldn't", "didn't", 'this', 'down', 'out', 'own', 'herself', 'in', 'shouldn', 'couldn', "won't", 'further', "it'd", 'only', 'until', 'against', 'being', 'off', "haven't", "it'll", 'their', 'don', 'aren', 'haven', "she'll", "you'll", 'these', 'the', 'our', "we're", 'or', 'those', "we'll", 'weren', "it's", "weren't", 'won', 'from', 'each', 'whom', 'because', 'hers', 'does', "isn't", 'it', 'do', 'into', 'his', 'up', 'isn', 'on', 'very', 'you', 'were', 'why', "aren't", "hadn't", 'mustn', 'had', 't', 'has', 'what', 'some', 'nor', "i'll", "they'd", 'below', 'd', 'yourselves', 'shan', "hasn't", 'ain', 'more', 'while', 'how', 'needn', "wasn't", 'no', "they're", 'they', 'themselves', 'after', 'mightn', 'by', 'doesn', 'but', "needn't", 'over', 'just', 'i', 'itself', 'didn', "don't", 'such', 'under', 'through', 'here', 'your', 'am', 'that', 'y', 'with', 'yourself', 'during', 'should', 

[nltk_data] Downloading package stopwords to /Users/mac/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
def remove_stop_words(data):
    stop_words = set(stopwords.words('english'))
    stop_stop_words = {"no","not"}
    
    stop_words = stop_words - stop_stop_words
    
    words = word_tokenize(str(data))
    
    new_text = ""
    for w in words:
        if w not in stop_words and len(w) > 1:
            new_text = new_text + " " + w
            
    return new_text

In [7]:
def remove_punctuations(data):
    symbols = " !()-[]{};:'\,<>./?@#$%^&*_~ '"""  
    
    for i in range(len(symbols)):
        data = np.char.replace(data,symbols[i]," ")
        data = np.char.replace(data,"  "," ")
    
    data = np.char.replace(data,",", " ")
    
    return data
            

In [8]:
def stemming(data):
    stemmer = PorterStemmer()
    
    tokens = word_tokenize(str(data))
    new_text = ""
    for w in tokens:
        new_text = new_text + " " + stemmer.stem(w)
        
    return new_text

def lemmatizing(data):
    lemmatizer = WordNetLemmatizer()
    
    tokens = word_tokenize(str(data))
    new_text = ""
    for w in tokens:
        new_text = new_text + " " + lemmatizer.lemmatize(w)
               
    return new_text

In [9]:
def convert_numbers(data):
    
    tokens = word_tokenize(str(data))
    new_text = ""
    for w in tokens:
        if w.isdigit():
            if int(w) < 1000000000:
                w = num2words(w)
            else:
                w = " "
        new_text = new_text + " " + w
            
    new_text = np.char.replace(new_text,"-"," ")
        
    return new_text

In [10]:
def remove_url_from_string(data):
    words = word_tokenize(str(data))
    new_text = ""

    
    for w in words:
        w = re.sub(r'https?:\/\/.*[\r\n]*',"",str(w),flags=re.MULTILINE)
        w = re.sub(r'http?:\/\/.*[\r\n]*',"",str(w),flags=re.MULTILINE)
        
        new_text = new_text + " " + w
        
    return new_text

In [11]:
def preprocess(data, word_root_algo='lemm'):
    
    data = remove_one_letter_word(data)
    data = convert_lower_case(data)
    data = remove_stop_words(data)
    data = stemming(data)
    data = remove_punctuations(data)
    data = convert_numbers(data)
    data = remove_url_from_string(data)
    
    if word_root_algo =='lemm':
        data = lemmatizing(data)
    else: 
        data = stemming(data)
        
    
    data = remove_punctuations(data)
    data = remove_stop_words(data)
    
    return data

In [12]:
#nltk.download('punkt')
#nltk.download('wordnet')
data['report_text_lemm'] = data['html_content_v6'].apply(lambda x: preprocess(x,"lemm"))

In [13]:
data['report_text_stemm'] = data['html_content_v6'].apply(lambda x: preprocess(x,"stemm"))

In [14]:
data.head()

,Unnamed: 0,id,date,short_url,title,text_title,full_url,html_content,html_content_v6,report_text_lemm,report_text_stemm
0,1201,1202,2022-02-24,2022_02_24,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty f...,russian offen campaign assess februari twenti...
1,1366,1367,2022-02-25,2022_02_25,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty f...,russian offen campaign assess februari twenti...
2,181,182,2022-02-26,2022_02_26,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty s...,russian offen campaign assess februari twenti...
3,197,198,2022-02-27,2022_02_27,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty s...,russian offen campaign assess februari twenti...
4,1106,1107,2022-02-28,2022_02_28,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty e...,russian offen campaign assess februari twenti...


In [15]:
data['report_text_lemm'][0]

' russian offens campaign ass februari twenty four two thousand twenty five isw option download page print pageshareskip contentskip content donat menu menu iswabout iswour capabilitiesisw compar advantageour historyimprov nation secur debatewho aremeet teamcareersjoin missionopen positionsmap roomisw produc world premier open sourc conflict map see mapsanalysisanalysisth gener jack kean center nation securitymap roomdiv isw complet map catalogbrief roomvideo podcast interact contentresearch librarybrows isw entir bodi workteams portfoliosrussia ukrainemiddl eastchina taiwandefens europeadversari ententecontemporari futur wargeospati intelligencecognit warfareeducationeducationth gener david petraeu center emerg leadershertog war studiesisw premier educ program undergraduatesfellowshipsenrich advanc opportun entry level mid car professionalsinternshipslaunch career isw internship open sourc intellig geospati intellig cognit warfar communicationsresearch librarybrows isw publish work cu

In [16]:
docs = data['report_text_lemm'].tolist()

In [17]:
#data.to_csv("")

CountVectorizer

In [18]:
cv = CountVectorizer(max_df=0.98,min_df=2)
word_count_vector = cv.fit_transform(docs)

word_count_vector.shape

(1464, 14252)

In [19]:
data.head(10)

,Unnamed: 0,id,date,short_url,title,text_title,full_url,html_content,html_content_v6,report_text_lemm,report_text_stemm
0,1201,1202,2022-02-24,2022_02_24,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty f...,russian offen campaign assess februari twenti...
1,1366,1367,2022-02-25,2022_02_25,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty f...,russian offen campaign assess februari twenti...
2,181,182,2022-02-26,2022_02_26,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty s...,russian offen campaign assess februari twenti...
3,197,198,2022-02-27,2022_02_27,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty s...,russian offen campaign assess februari twenti...
4,1106,1107,2022-02-28,2022_02_28,"Russian Offensive Campaign Assessment, Februar...","Russian Offensive Campaign Assessment, Februar...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, Februar...",russian offens campaign ass februari twenty e...,russian offen campaign assess februari twenti...
5,547,548,2022-03-01,2022_03_01,"Russian Offensive Campaign Assessment, March 1...","Russian Offensive Campaign Assessment, March 1...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, March 1...",russian offens campaign ass march isw option ...,russian offen campaign assess march isw optio...
6,998,999,2022-03-02,2022_03_02,"Russian Offensive Campaign Assessment, March 2...","Russian Offensive Campaign Assessment, March 2...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, March 2...",russian offens campaign ass march isw option ...,russian offen campaign assess march isw optio...
7,838,839,2022-03-03,2022_03_03,"Russian Offensive Campaign Assessment, March 3...","Russian Offensive Campaign Assessment, March 3...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, March 3...",russian offens campaign ass march isw option ...,russian offen campaign assess march isw optio...
8,1454,1455,2022-03-04,2022_03_04,"Russian Offensive Campaign Assessment, March 4...","Russian Offensive Campaign Assessment, March 4...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, March 4...",russian offens campaign ass march isw option ...,russian offen campaign assess march isw optio...
9,1110,1111,2022-03-05,2022_03_05,"Russian Offensive Campaign Assessment, March 5...","Russian Offensive Campaign Assessment, March 5...",https://understandingwar.org/research/russia-u...,"<!DOCTYPE html>\n<html lang=""en-US"">\n<head>\n...","Russian Offensive Campaign Assessment, March 5...",russian offens campaign ass march isw option ...,russian offen campa

In [20]:
word_count_vector

<1464x14252 sparse matrix of type '<class 'numpy.int64'>'
	with 939878 stored elements in Compressed Sparse Row format>

In [21]:
with open(r"count_vectorizer_v1.pkl", "wb") as handle:
    pickle.dump(cv, handle)

TF-IDF

In [22]:
tfidf_transformer = TfidfTransformer(smooth_idf = True, use_idf = True)
tfidf_transformer.fit(word_count_vector)

TfidfTransformer()

In [23]:
with open("tfidf_transformer.pkl","wb") as handle:
    pickle.dump(tfidf_transformer, handle)

In [24]:
df_idf = pd.DataFrame(tfidf_transformer.idf_, index = cv.get_feature_names_out(),columns=["idf_weights"])
df_idf.sort_values(by=['idf_weights'])

,idf_weights
territori,1.024880
assessmentread,1.024880
missionopen,1.024880
captur,1.024880
defens,1.024880
...,...
lazutkin,6.680173
layout,6.680173
layoff,6.680173
lax,6.680173


In [25]:
tf_idf_vector = tfidf_transformer.transform(word_count_vector)

In [26]:
tf_idf_vector

<1464x14252 sparse matrix of type '<class 'numpy.float64'>'
	with 939878 stored elements in Compressed Sparse Row format>

Transformation with fitted models

In [27]:
tfidf = pickle.load(open("tfidf_transformer.pkl","rb"))
cv = pickle.load(open("count_vectorizer_v1.pkl","rb"))

In [28]:
feature_names = cv.get_feature_names_out()

In [29]:
feature_names

array(['000km', '00pm', '02owivyrp402xkkwk', ..., 'программа',
       'підрозділи', 'сухопутный'], dtype=object)

In [30]:
def sort_coo(coo_matrix):
    
    tuples = zip(coo_matrix.col, coo_matrix.data)
    
    return sorted(tuples,key=lambda x: (x[1], x[0]), reverse=True)

def extract_topn_from_vector(feature_names, sorted_items, topn=10):
    
    sorted_items = sorted_items[:topn]
    
    score_vals = []
    feature_vals = []
    
    for idx, score in sorted_items:
        
        score_vals.append(round(score,3))
        feature_vals.append(feature_names[idx])
        
    results = {}
    
    for idx in range(len(feature_vals)):
        results[feature_vals[idx]] = score_vals[idx]
    
    return results


def convert_doc_to_vector(doc):
    feature_names = cv.get_feature_names_out()
    top_n = 10
    tf_idf_vector = tfidf.transform(cv.transform([doc]))
    
    sorted_items = sort_coo(tf_idf_vector.tocoo())
    
    keywords = extract_topn_from_vector(feature_names, sorted_items, top_n)
    
    return keywords

In [31]:
data['keywords'] = data['report_text_stemm'].apply(lambda x: convert_doc_to_vector(x))

In [32]:
data['keywords'][0]

{'twenti': 0.897,
 'februari': 0.205,
 'hundr': 0.142,
 'near': 0.071,
 'motor': 0.07,
 'rifl': 0.069,
 'thirti': 0.059,
 'percent': 0.052,
 'pokrovsk': 0.052,
 'eighti': 0.051}

In [33]:
data_vectorised = tf_idf_vector.toarray()

In [34]:
vectors_df = pd.DataFrame(data_vectorised)

In [35]:
#vectors_df.to_csv("")